# JAX Frameworks: Module Registration & Container Tracking Summary

A comprehensive reference guide detailing how native JAX, Flax NNX, and Equinox register their tracked modules, manage mutability state, handle standard versus framework-specific containers, and register custom tracking behaviors.

---

## Overview Matrix


| Feature | Native JAX | Flax NNX | Equinox |
| :--- | :--- | :--- | :--- |
| **Registration Strategy** | **No base class.** Uses explicit registration via `jax.tree_util.register_pytree_node` or `@jax.tree_util.register_dataclass`. | **Concrete base class.** Uses `nnx.Module` (which inherits from `nnx.Pytree`). Auto-registers upon subclassing. | **Concrete base class.** Uses `eqx.Module` (which converts subclasses into immutable Python dataclasses and auto-registers them). |
| **Mutability State** | Purely **functional/immutable**. | **Stateful/mutable** objects at runtime. Uses `nnx.split()` to transform into pure JAX structures. | Purely **immutable** dataclasses. Modifying state requires functional updates (e.g., `eqx.tree_at`). |
| **Automatically Tracked Native Containers** | `list`, `tuple`, `dict`, `NamedTuple`, `None`. | **None (for NNX variables).** Raw Python `list` and `dict` elements hide NNX state from JAX tracing. | `list`, `tuple`, `dict`, `NamedTuple`, `None`. Fully supported natively. |
| **Framework-Specific Tracked Containers** | N/A | Must use **`nnx.List`** and **`nnx.Dict`** wrapper types to hold sub-modules or variables. | N/A (Standard Python containers work seamlessly out of the box). |
| **Tracking Custom Classes / Variables** | Call `jax.tree_util.register_pytree_node(Class, flatten_fn, unflatten_fn)` to explicitly declare dynamic leaves and auxiliary metadata. | Inherit from **`nnx.Pytree`** for complex structures, or wrap custom leaf datatypes with the **`@nnx.register_data_type`** decorator. | Inherit from **`eqx.Module`**. Ensure all tracked fields have Python **type annotations**, which automatically converts them into a dataclass. |

---

## Code Boilerplate: Tracking Variables in Custom Classes

### 1. Native JAX
To make a custom class recognizable to JAX transformations, you must manually specify how to unpack its parameters (leaves) and non-array configurations (auxiliary metadata).

```python
import jax

class CustomLayer:
    def __init__(self, weight, bias, activation_name):
        self.weight = weight
        self.bias = bias
        self.activation_name = activation_name  # Static metadata

# 1. Define how to flatten the class into dynamic leaves and static metadata
def custom_layer_flatten(node):
    children = (node.weight, node.bias)  # Dynamic elements tracked by JAX
    aux_data = (node.activation_name,)   # Static elements ignored by JAX operations
    return (children, aux_data)

# 2. Define how to reconstruct the class from its pieces
def custom_layer_unflatten(aux_data, children):
    weight, bias = children
    activation_name, = aux_data
    return CustomLayer(weight, bias, activation_name)

# 3. Register the class explicitly with JAX
jax.tree_util.register_pytree_node(
    CustomLayer, 
    custom_layer_flatten, 
    custom_layer_unflatten
)
```

### 2. Flax NNX
Flax NNX dynamically manages stateful graphs. Custom structures can act as PyTree containers by inheriting from the `nnx.Pytree` base class.

```python
from flax import nnx
import jax.numpy as jnp

# Subclassing nnx.Pytree handles the JAX registration automatically
class CustomStateContainer(nnx.Pytree):
    def __init__(self, size: int):
        # Explicitly wrap parameters in NNX Variable types so they are tracked
        self.weights = nnx.Param(jnp.ones((size, size)))
        self.bias = nnx.Param(jnp.zeros((size,)))
        self.config_str = "relu"  # Raw non-variable types are treated as static metadata
```

### 3. Equinox
Equinox completely removes custom flattening functions and variable wrappers by piggybacking off native Python dataclass annotations.

```python
import equinox as eqx
import jax

# Inheriting automatically makes this a JAX-registered dataclass
class CustomBlock(eqx.Module):
    # Every field MUST be type-annotated to be picked up by the dataclass engine
    weights: jax.Array
    bias: jax.Array
    layer_name: str  # Filtered transformations (like eqx.filter_jit) handle this static string automatically

    def __init__(self, size: int, key: jax.random.PRNGKey):
        w_key, b_key = jax.random.split(key)
        self.weights = jax.random.normal(w_key, (size, size))
        self.bias = jax.random.normal(b_key, (size,))
        self.layer_name = "dense_block"
```

---

## Key Implementation Takeaways

* **Choose Native JAX** if you are writing low-level algorithmic primitives and want absolute control over your tracking logic without framework baggage.
* **Choose Flax NNX** if you prefer a PyTorch-style, stateful, or imperative object-oriented workflow, but remain mindful of utilizing `nnx.List`/`nnx.Dict` when managing grouped layers.
* **Choose Equinox** if you value strict functional purity, minimal architectural layers, and want to leverage native Python structures without breaking the JAX compilation chain.

